# T21 - 海外银行负债成本分析

## 项目概述

本项目研究全球主要经济体的银行负债成本变化趋势，分析利率周期对银行负债成本的影响。

### 主要功能
- 多国负债成本对比分析
- 利率周期影响分析
- 滞后性研究
- 可视化报告

### 研究对象
| 地区 | 主要指标 |
|------|----------|
| 美国 | 联邦基金利率、银行负债成本 |
| 欧元区 | 欧央行政策利率、银行负债成本 |
| 日本 | 日银政策利率、银行负债成本 |
| 新加坡 | 新加坡政策利率、银行负债成本 |

### 核心分析维度
1. **利率时间序列对比** - 政策利率 vs 银行负债成本
2. **利率差异分析** - 正常利率期 vs 低利率期
3. **利率变化率对比** - 政策利率变化速度 vs 负债成本调整速度
4. **滞后性分析** - 政策利率传导时滞研究

---
## 1. 环境配置

导入必要的依赖库并配置运行环境。

In [ ]:
# 导入标准库
import os
import warnings
from datetime import datetime

# 导入数据处理库
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# 导入可视化库
import matplotlib.pyplot as plt
import seaborn as sns

# 忽略警告
warnings.filterwarnings('ignore')

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 导入项目配置（如果存在）
try:
    from config import (
        COUNTRIES, DATA_FILE, OUTPUT_DIR, COLORS, FIGURE_CONFIG,
        RATE_CHANGE_THRESHOLD, MAX_LAG_PERIODS, OUTPUT_FILES,
        ensure_output_dir, get_output_path
    )
    print("配置文件加载成功")
except ImportError:
    print("未找到配置文件，使用默认配置")
    # 默认配置
    DATA_FILE = 'complete_liability_cost_analysis_with_raw_data.xlsx'
    OUTPUT_DIR = './output'
    COLORS = {
        'primary': ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
        'upward_policy': 'blue',
        'upward_liability': 'lightblue',
        'downward_policy': 'red',
        'downward_liability': 'lightcoral'
    }
    FIGURE_CONFIG = {'dpi': 300, 'bbox_inches': 'tight'}
    RATE_CHANGE_THRESHOLD = 0.1
    MAX_LAG_PERIODS = 3
    
    def ensure_output_dir():
        if not os.path.exists(OUTPUT_DIR):
            os.makedirs(OUTPUT_DIR)
        return OUTPUT_DIR
    
    def get_output_path(filename):
        return os.path.join(OUTPUT_DIR, filename)

print(f"当前工作目录: {os.getcwd()}")
print(f"数据文件: {DATA_FILE}")
print(f"输出目录: {OUTPUT_DIR}")

---
## 2. 数据获取

加载Excel数据文件并解析数据结构。

In [ ]:
def load_excel_data(file_path):
    """
    加载Excel数据文件
    
    Parameters:
    -----------
    file_path : str
        Excel文件路径
    
    Returns:
    --------
    dict : {工作表名称: DataFrame}
    """
    try:
        xl = pd.ExcelFile(file_path)
        print(f"工作表名称: {xl.sheet_names}")
        
        data_dict = {}
        for sheet_name in xl.sheet_names:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
            data_dict[sheet_name] = df
            print(f"\n{sheet_name}:")
            print(f"  - 数据形状: {df.shape}")
            print(f"  - 列名: {list(df.columns)}")
        
        return data_dict
        
    except Exception as e:
        print(f"读取文件时出错: {e}")
        return None

# 加载数据
data_dict = load_excel_data(DATA_FILE)

if data_dict:
    print(f"\n成功加载 {len(data_dict)} 个工作表")

In [ ]:
# 查看数据样例
if data_dict:
    for sheet_name, df in data_dict.items():
        print(f"\n{'='*60}")
        print(f"工作表: {sheet_name}")
        print('='*60)
        display(df.head(10))

---
## 3. 数据处理

数据清洗、按国家/指标分类处理。

In [ ]:
def clean_country_data(df, country_config):
    """
    清洗单个国家的数据
    
    Parameters:
    -----------
    df : DataFrame
        原始数据
    country_config : dict
        国家配置信息
    
    Returns:
    --------
    DataFrame : 清洗后的数据
    """
    # 提取关键列
    cols_to_keep = [
        country_config['policy_rate_col'],
        country_config['liability_cost_col'],
        country_config['bond_yield_col'],
        '年份',
        '时期分类'
    ]
    
    # 只保留存在的列
    existing_cols = [col for col in cols_to_keep if col in df.columns]
    df_clean = df[existing_cols].copy()
    
    # 删除空值
    df_clean = df_clean.dropna()
    
    # 按年份排序
    if '年份' in df_clean.columns:
        df_clean = df_clean.sort_values('年份')
    
    return df_clean

def calculate_rate_changes(df, policy_col, liability_col, bond_col):
    """
    计算利率变化和趋势
    
    Parameters:
    -----------
    df : DataFrame
        数据框
    policy_col : str
        政策利率列名
    liability_col : str
        负债成本列名
    bond_col : str
        国债收益率列名
    
    Returns:
    --------
    DataFrame : 添加变化列的数据框
    """
    df = df.copy()
    
    # 计算利率变化
    df['policy_rate_change'] = df[policy_col].diff()
    df['liability_cost_change'] = df[liability_col].diff()
    df['bond_yield_change'] = df[bond_col].diff()
    
    # 识别趋势
    df['rate_trend'] = np.where(
        df['policy_rate_change'] > RATE_CHANGE_THRESHOLD, '上行',
        np.where(df['policy_rate_change'] < -RATE_CHANGE_THRESHOLD, '下行', '持平')
    )
    
    return df

# 处理所有国家数据
processed_data = {}

if data_dict:
    # 国家配置
    countries_config = {
        '美国': {
            'sheet_name': '美国完整原始数据',
            'policy_rate_col': '联邦基金利率(%)',
            'liability_cost_col': '负债成本_总负债口径(%)',
            'bond_yield_col': '美国10年国债收益率(%)'
        },
        '欧元区': {
            'sheet_name': '欧元区完整原始数据',
            'policy_rate_col': 'ECB主要再融资利率(%)',
            'liability_cost_col': '负债成本_总负债口径(%)',
            'bond_yield_col': '德国10年国债收益率(%)'
        },
        '日本': {
            'sheet_name': '日本完整原始数据',
            'policy_rate_col': 'BOJ政策利率(%)',
            'liability_cost_col': '负债成本_总负债口径(%)',
            'bond_yield_col': '日本10年国债收益率(%)'
        },
        '新加坡': {
            'sheet_name': '新加坡完整原始数据',
            'policy_rate_col': 'MAS政策利率(%)',
            'liability_cost_col': '负债成本_总负债口径(%)',
            'bond_yield_col': '新加坡10年国债收益率(%)'
        }
    }
    
    for country, config in countries_config.items():
        sheet_name = config['sheet_name']
        if sheet_name in data_dict:
            df = data_dict[sheet_name]
            df_clean = clean_country_data(df, config)
            df_processed = calculate_rate_changes(
                df_clean,
                config['policy_rate_col'],
                config['liability_cost_col'],
                config['bond_yield_col']
            )
            processed_data[country] = {
                'data': df_processed,
                'config': config
            }
            print(f"{country}: 处理完成，共 {len(df_processed)} 条记录")

In [ ]:
# 查看处理后的数据样例
for country, data in processed_data.items():
    print(f"\n{'='*60}")
    print(f"{country} - 处理后数据")
    print('='*60)
    display(data['data'].head())

---
## 4. 核心逻辑

包含跨国对比分析、利率周期分析、滞后性分析、相关性计算等核心功能。

### 4.1 跨国对比分析

In [ ]:
def create_cross_country_comparison_chart(data_dict, output_dir):
    """
    创建跨国对比分析图表
    
    Parameters:
    -----------
    data_dict : dict
        数据字典
    output_dir : str
        输出目录
    """
    # 查找跨国汇总对比分析数据
    cross_country_sheet = None
    for sheet_name in data_dict.keys():
        if '跨国汇总' in sheet_name or '对比分析' in sheet_name:
            cross_country_sheet = sheet_name
            break
    
    if not cross_country_sheet:
        print("未找到跨国汇总对比分析数据")
        return
    
    df = data_dict[cross_country_sheet]
    
    # 过滤负债成本数据
    liability_data = df[df['指标类型'].str.contains('负债成本', na=False)]
    
    if liability_data.empty:
        print("未找到负债成本数据")
        return
    
    # 分组数据
    total_liability = liability_data[liability_data['指标类型'] == '负债成本_总负债口径']
    deposit_liability = liability_data[liability_data['指标类型'] == '负债成本_存款口径']
    
    # 创建图表
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    x = np.arange(len(total_liability))
    width = 0.35
    
    # 子图1：总负债口径对比
    ax1 = axes[0, 0]
    bars1 = ax1.bar(x - width/2, total_liability['正常利率期平均(%)'], width,
                    label='正常利率期', alpha=0.8, color='steelblue')
    bars2 = ax1.bar(x + width/2, total_liability['低利率期平均(%)'], width,
                    label='低利率期', alpha=0.8, color='lightcoral')
    ax1.set_title('负债成本_总负债口径 - 正常vs低利率期对比')
    ax1.set_ylabel('成本 (%)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(total_liability['国家'], rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 添加数值标签
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=9)
    
    # 子图2：存款口径对比
    ax2 = axes[0, 1]
    if not deposit_liability.empty:
        bars3 = ax2.bar(x - width/2, deposit_liability['正常利率期平均(%)'], width,
                        label='正常利率期', alpha=0.8, color='darkgreen')
        bars4 = ax2.bar(x + width/2, deposit_liability['低利率期平均(%)'], width,
                        label='低利率期', alpha=0.8, color='lightgreen')
        ax2.set_title('负债成本_存款口径 - 正常vs低利率期对比')
        ax2.set_ylabel('成本 (%)')
        ax2.set_xticks(x)
        ax2.set_xticklabels(deposit_liability['国家'], rotation=45)
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # 子图3：成本降幅对比
    ax3 = axes[1, 0]
    if '成本/利率降幅(bp)' in total_liability.columns:
        bars5 = ax3.bar(x, total_liability['成本/利率降幅(bp)'], alpha=0.8, color='orange')
        ax3.set_title('总负债口径 - 成本/利率降幅对比')
        ax3.set_ylabel('降幅 (bp)')
        ax3.set_xticks(x)
        ax3.set_xticklabels(total_liability['国家'], rotation=45)
        ax3.grid(True, alpha=0.3)
    
    # 子图4：波动幅度对比
    ax4 = axes[1, 1]
    if '波动幅度(bp)' in total_liability.columns:
        bars6 = ax4.bar(x, total_liability['波动幅度(bp)'], alpha=0.8, color='purple')
        ax4.set_title('总负债口径 - 波动幅度对比')
        ax4.set_ylabel('波动幅度 (bp)')
        ax4.set_xticks(x)
        ax4.set_xticklabels(total_liability['国家'], rotation=45)
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, '图表1_跨国负债成本对比分析.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"图表已保存: {output_path}")

# 执行跨国对比分析
ensure_output_dir()
if data_dict:
    create_cross_country_comparison_chart(data_dict, OUTPUT_DIR)

### 4.2 利率周期分析

In [ ]:
def analyze_rate_cycles(processed_data):
    """
    分析利率周期（上行/下行）
    
    Parameters:
    -----------
    processed_data : dict
        处理后的数据字典
    
    Returns:
    --------
    dict : 周期分析结果
    """
    cycle_results = {}
    
    for country, data in processed_data.items():
        df = data['data']
        
        # 统计上行和下行周期
        upward_periods = df[df['rate_trend'] == '上行']
        downward_periods = df[df['rate_trend'] == '下行']
        
        cycle_results[country] = {
            'upward': {
                'count': len(upward_periods),
                'avg_policy_change': upward_periods['policy_rate_change'].mean() if len(upward_periods) > 0 else 0,
                'avg_liability_change': upward_periods['liability_cost_change'].mean() if len(upward_periods) > 0 else 0
            },
            'downward': {
                'count': len(downward_periods),
                'avg_policy_change': downward_periods['policy_rate_change'].mean() if len(downward_periods) > 0 else 0,
                'avg_liability_change': downward_periods['liability_cost_change'].mean() if len(downward_periods) > 0 else 0
            }
        }
    
    return cycle_results

def create_cycle_analysis_chart(cycle_results, output_dir):
    """
    创建周期分析图表
    """
    countries = list(cycle_results.keys())
    
    upward_policy = [cycle_results[c]['upward']['avg_policy_change'] for c in countries]
    upward_liability = [cycle_results[c]['upward']['avg_liability_change'] for c in countries]
    downward_policy = [abs(cycle_results[c]['downward']['avg_policy_change']) for c in countries]
    downward_liability = [abs(cycle_results[c]['downward']['avg_liability_change']) for c in countries]
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    x = np.arange(len(countries))
    width = 0.35
    
    # 上行周期
    ax1 = axes[0]
    bars1 = ax1.bar(x - width/2, upward_policy, width, label='政策利率上行变化', 
                    alpha=0.8, color='blue')
    bars2 = ax1.bar(x + width/2, upward_liability, width, label='负债成本上行变化', 
                    alpha=0.8, color='lightblue')
    ax1.set_title('利率上行周期 - 政策利率与负债成本变化对比')
    ax1.set_ylabel('变化幅度 (%)')
    ax1.set_xlabel('国家/地区')
    ax1.set_xticks(x)
    ax1.set_xticklabels(countries)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                ha='center', va='bottom')
    for bar in bars2:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                ha='center', va='bottom')
    
    # 下行周期
    ax2 = axes[1]
    bars3 = ax2.bar(x - width/2, downward_policy, width, label='政策利率下行变化', 
                    alpha=0.8, color='red')
    bars4 = ax2.bar(x + width/2, downward_liability, width, label='负债成本下行变化', 
                    alpha=0.8, color='lightcoral')
    ax2.set_title('利率下行周期 - 政策利率与负债成本变化对比')
    ax2.set_ylabel('变化幅度 (%)')
    ax2.set_xlabel('国家/地区')
    ax2.set_xticks(x)
    ax2.set_xticklabels(countries)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    for bar in bars3:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                ha='center', va='bottom')
    for bar in bars4:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}', 
                ha='center', va='bottom')
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, '上行下行周期分析对比.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"图表已保存: {output_path}")

# 执行周期分析
if processed_data:
    cycle_results = analyze_rate_cycles(processed_data)
    create_cycle_analysis_chart(cycle_results, OUTPUT_DIR)
    
    # 打印周期分析结果
    print("\n=== 利率周期分析结果 ===")
    for country, result in cycle_results.items():
        print(f"\n{country}:")
        print(f"  上行周期: {result['upward']['count']}次, 政策利率平均上升{result['upward']['avg_policy_change']:.2f}%, 负债成本平均上升{result['upward']['avg_liability_change']:.2f}%")
        print(f"  下行周期: {result['downward']['count']}次, 政策利率平均下降{abs(result['downward']['avg_policy_change']):.2f}%, 负债成本平均下降{abs(result['downward']['avg_liability_change']):.2f}%")

### 4.3 滞后性分析

In [ ]:
def analyze_lag_correlation(processed_data, max_lag=3):
    """
    分析政策利率与负债成本的滞后相关性
    
    Parameters:
    -----------
    processed_data : dict
        处理后的数据字典
    max_lag : int
        最大滞后期数
    
    Returns:
    --------
    dict : 滞后性分析结果
    """
    lag_results = {}
    
    for country, data in processed_data.items():
        df = data['data']
        config = data['config']
        
        policy_col = config['policy_rate_col']
        liability_col = config['liability_cost_col']
        
        correlations = []
        actual_max_lag = min(max_lag, len(df) - 2)
        
        for lag in range(0, actual_max_lag + 1):
            try:
                if lag == 0:
                    corr, p_value = pearsonr(
                        df[policy_col].dropna(),
                        df[liability_col].dropna()
                    )
                else:
                    policy_shifted = df[policy_col].iloc[:-lag]
                    liability_current = df[liability_col].iloc[lag:]
                    if len(policy_shifted) > 2 and len(liability_current) > 2:
                        corr, p_value = pearsonr(policy_shifted, liability_current)
                    else:
                        corr, p_value = 0, 1
                
                correlations.append({
                    'lag': lag,
                    'correlation': corr,
                    'p_value': p_value
                })
            except Exception as e:
                correlations.append({
                    'lag': lag,
                    'correlation': 0,
                    'p_value': 1
                })
        
        # 找出最佳滞后期
        best_corr = max(correlations, key=lambda x: abs(x['correlation']))
        
        lag_results[country] = {
            'correlations': correlations,
            'best_lag': best_corr['lag'],
            'best_correlation': best_corr['correlation']
        }
    
    return lag_results

def create_lag_correlation_chart(lag_results, output_dir):
    """
    创建滞后相关性图表
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for i, (country, result) in enumerate(lag_results.items()):
        ax = axes[i // 2, i % 2]
        correlations = result['correlations']
        lags = [c['lag'] for c in correlations]
        corr_values = [c['correlation'] for c in correlations]
        
        ax.plot(lags, corr_values, 'o-', linewidth=2, markersize=8,
               color=colors[i], label=country)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_title(f'{country} - 政策利率与负债成本滞后相关性')
        ax.set_xlabel('滞后期数')
        ax.set_ylabel('相关系数')
        ax.grid(True, alpha=0.3)
        
        # 标记最佳滞后期
        best_lag = result['best_lag']
        best_corr = result['best_correlation']
        ax.scatter(best_lag, best_corr, color='red', s=100, zorder=5)
        ax.text(best_lag, best_corr + 0.05,
               f'最佳滞后期: {best_lag}',
               ha='center', fontsize=10,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, '滞后性分析汇总_相关性对比.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"图表已保存: {output_path}")

# 执行滞后性分析
if processed_data:
    lag_results = analyze_lag_correlation(processed_data, MAX_LAG_PERIODS)
    create_lag_correlation_chart(lag_results, OUTPUT_DIR)
    
    # 打印滞后性分析结果
    print("\n=== 滞后性分析结果 ===")
    for country, result in lag_results.items():
        print(f"{country}: 最佳滞后期 {result['best_lag']}期, 最高相关性 {result['best_correlation']:.3f}")

### 4.4 时间序列对比

In [ ]:
def create_time_series_chart(processed_data, output_dir):
    """
    创建时间序列对比图表
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    for i, (country, data) in enumerate(processed_data.items()):
        df = data['data']
        config = data['config']
        
        ax = axes[i // 2, i % 2]
        
        # 绘制政策利率
        ax.plot(df['年份'], df[config['policy_rate_col']], 'o-', linewidth=3, 
               markersize=6, label='政策利率', color='blue', alpha=0.8)
        
        # 绘制负债成本
        ax.plot(df['年份'], df[config['liability_cost_col']], 's-', linewidth=3, 
               markersize=6, label='银行负债成本', color='red', alpha=0.8)
        
        ax.set_title(f'{country} - 政策利率与银行负债成本对比')
        ax.set_xlabel('年份')
        ax.set_ylabel('利率 (%)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 标注上行和下行周期
        for idx, row in df.iterrows():
            if row['rate_trend'] == '上行':
                ax.axvspan(row['年份']-0.5, row['年份']+0.5, alpha=0.2, color='green')
            elif row['rate_trend'] == '下行':
                ax.axvspan(row['年份']-0.5, row['年份']+0.5, alpha=0.2, color='orange')
    
    plt.tight_layout()
    output_path = os.path.join(output_dir, '各国政策利率与负债成本时间序列对比.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"图表已保存: {output_path}")

# 执行时间序列分析
if processed_data:
    create_time_series_chart(processed_data, OUTPUT_DIR)

---
## 5. 执行与测试

执行主流程并生成分析结论。

In [ ]:
def generate_conclusions(cycle_results, lag_results, output_dir):
    """
    生成分析结论报告
    
    Parameters:
    -----------
    cycle_results : dict
        周期分析结果
    lag_results : dict
        滞后性分析结果
    output_dir : str
        输出目录
    """
    conclusions = []
    conclusions.append("=== 银行负债成本滞后性分析结论 ===\n")
    
    # 各国分析结论
    for country in cycle_results.keys():
        lag_result = lag_results.get(country, {})
        cycle_result = cycle_results.get(country, {})
        
        conclusions.append(f"{country}:")
        conclusions.append(f"  - 最佳滞后期: {lag_result.get('best_lag', 'N/A')}期")
        conclusions.append(f"  - 最高相关性: {lag_result.get('best_correlation', 0):.3f}")
        
        upward = cycle_result.get('upward', {})
        downward = cycle_result.get('downward', {})
        
        conclusions.append(f"  - 上行周期分析: 政策利率平均上升{upward.get('avg_policy_change', 0):.2f}%, "
                          f"负债成本平均上升{upward.get('avg_liability_change', 0):.2f}%")
        conclusions.append(f"  - 下行周期分析: 政策利率平均下降{abs(downward.get('avg_policy_change', 0)):.2f}%, "
                          f"负债成本平均下降{abs(downward.get('avg_liability_change', 0)):.2f}%")
        conclusions.append("")
    
    # 总体结论
    conclusions.append("=== 总体验证结论 ===")
    
    # 计算平均滞后期
    all_best_lags = [r['best_lag'] for r in lag_results.values()]
    avg_lag = np.mean(all_best_lags) if all_best_lags else 0
    conclusions.append(f"1. 平均最佳滞后期: {avg_lag:.1f}期")
    
    # 分析上下行周期敏感性
    all_upward_ratios = []
    all_downward_ratios = []
    
    for country, result in cycle_results.items():
        upward = result.get('upward', {})
        downward = result.get('downward', {})
        
        up_policy = upward.get('avg_policy_change', 0)
        up_liability = upward.get('avg_liability_change', 0)
        down_policy = downward.get('avg_policy_change', 0)
        down_liability = downward.get('avg_liability_change', 0)
        
        if up_policy != 0:
            all_upward_ratios.append(up_liability / up_policy)
        if down_policy != 0:
            all_downward_ratios.append(down_liability / down_policy)
    
    if all_upward_ratios and all_downward_ratios:
        avg_up_ratio = np.mean(all_upward_ratios)
        avg_down_ratio = np.mean(all_downward_ratios)
        conclusions.append(f"2. 上行周期负债成本/政策利率变化比例: {avg_up_ratio:.2f}")
        conclusions.append(f"3. 下行周期负债成本/政策利率变化比例: {avg_down_ratio:.2f}")
        
        if abs(avg_down_ratio) < abs(avg_up_ratio):
            conclusions.append("4. 下行周期中银行负债成本下降幅度小于政策利率下降幅度")
        else:
            conclusions.append("4. 上行和下行周期的敏感性基本相似")
    
    conclusions.append("\n=== 政策含义 ===")
    conclusions.append("1. 银行负债成本具有明显的滞后性，政策制定需要考虑传导时间")
    conclusions.append("2. 在利率下行周期，银行负债成本下降相对缓慢，银行存在一定的利息收入保护")
    conclusions.append("3. 不同国家的传导机制存在差异，需要区别化的货币政策工具")
    
    conclusion_text = "\n".join(conclusions)
    
    # 保存结论
    output_path = os.path.join(output_dir, '银行负债成本滞后性分析结论.txt')
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(conclusion_text)
    
    print(conclusion_text)
    print(f"\n结论已保存: {output_path}")
    
    return conclusion_text

# 执行结论生成
if processed_data and 'cycle_results' in dir() and 'lag_results' in dir():
    conclusion_text = generate_conclusions(cycle_results, lag_results, OUTPUT_DIR)

In [ ]:
# 汇总所有生成的文件
print("\n" + "="*60)
print("分析完成！生成的文件列表：")
print("="*60)

if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    for f in files:
        file_path = os.path.join(OUTPUT_DIR, f)
        file_size = os.path.getsize(file_path) / 1024  # KB
        print(f"  - {f} ({file_size:.1f} KB)")

---
## 6. 工具函数

可复用的辅助函数集合。

In [ ]:
def setup_chinese_font():
    """
    配置matplotlib中文字体支持
    """
    import matplotlib
    
    font_list = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
    
    for font in font_list:
        try:
            plt.rcParams['font.sans-serif'] = [font]
            plt.rcParams['axes.unicode_minus'] = False
            print(f"已设置中文字体: {font}")
            return True
        except:
            continue
    
    print("警告: 未找到合适的中文字体")
    return False


def safe_divide(numerator, denominator, default=0):
    """
    安全除法，避免除零错误
    
    Parameters:
    -----------
    numerator : float
        分子
    denominator : float
        分母
    default : float
        默认返回值
    
    Returns:
    --------
    float : 除法结果或默认值
    """
    if denominator == 0 or np.isnan(denominator):
        return default
    return numerator / denominator


def format_percentage(value, decimal=2):
    """
    格式化百分比显示
    
    Parameters:
    -----------
    value : float
        数值
    decimal : int
        小数位数
    
    Returns:
    --------
    str : 格式化后的字符串
    """
    return f"{value:.{decimal}f}%"


def validate_dataframe(df, required_columns):
    """
    验证DataFrame是否包含必需的列
    
    Parameters:
    -----------
    df : DataFrame
        待验证的数据框
    required_columns : list
        必需的列名列表
    
    Returns:
    --------
    tuple : (是否验证通过, 缺失的列名列表)
    """
    missing_cols = [col for col in required_columns if col not in df.columns]
    is_valid = len(missing_cols) == 0
    return is_valid, missing_cols


def get_data_summary(df):
    """
    获取数据摘要统计
    
    Parameters:
    -----------
    df : DataFrame
        数据框
    
    Returns:
    --------
    dict : 统计摘要
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    summary = {
        'shape': df.shape,
        'columns': list(df.columns),
        'numeric_stats': {}
    }
    
    for col in numeric_cols:
        summary['numeric_stats'][col] = {
            'mean': df[col].mean(),
            'std': df[col].std(),
            'min': df[col].min(),
            'max': df[col].max()
        }
    
    return summary


print("工具函数加载完成")

---
## 7. 总结

本项目完成了以下分析：

1. **跨国负债成本对比分析** - 比较美国、欧元区、日本、新加坡的银行负债成本
2. **利率周期分析** - 分析上行和下行周期中政策利率与负债成本的变化
3. **滞后性分析** - 研究政策利率对负债成本的传导时滞
4. **时间序列对比** - 展示各国政策利率与负债成本的历史变化趋势

### 主要发现

- 银行负债成本与政策利率高度相关
- 不同国家的传导机制存在差异
- 在利率下行周期，银行负债成本下降相对缓慢